# CAMS Radiation Data Download for Uganda

This notebook downloads radiation data from the Copernicus Atmosphere Monitoring Service (CAMS) for a grid of points covering Uganda. The data is essential for solar resource assessment and analysis across the country.

## Overview

The notebook performs the following tasks:
1. Creates a grid of points covering Uganda using a specified grid spacing
2. Filters points to include only those within Uganda's boundaries
3. Downloads radiation data for each point from CAMS
4. Saves the data in a structured CSV format

## Requirements

### CAMS API Access
1. Register for CAMS radiation service at [CAMS Radiation Service](https://atmosphere.copernicus.eu/)
2. Your registered email will be required when running this notebook
3. The API has a limit of 100 requests per day

### Software Requirements
- Python packages: numpy, pandas, geopandas, shapely
- Uganda shapefile (provided in the repository)
- SuSSE package (internal package for API interactions)

## Data Format

The downloaded data will be saved in a CSV file with the following structure:
- `latitude`: The latitude coordinate of the point
- `longitude`: The longitude coordinate of the point
- `timestamp`: Time of measurement in ISO format
- Additional columns for radiation parameters from CAMS

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import time
import os
import geopandas as gpd
from shapely.geometry import Point
from collections import deque
from datetime import timedelta

# Import custom utilities
from susse import CAMSClient
from download_utils import (
    check_existing_coordinates, 
    should_skip_coordinate, 
    add_coordinate_to_existing,
    print_download_summary,
    show_data_preview
)

# Configuration parameters
GRID_SPACING_DEGREES = 0.1  # Grid spacing in degrees
UGANDA_SHAPEFILE = "Uganda_shape_files/gadm41_UGA_0.shp"
API_DAILY_LIMIT = 100  
API_COOLDOWN = 30  

# Time configuration
start_date = datetime(2024, 1, 1, tzinfo=timezone.utc)
end_date = datetime(2024, 12, 31, 23, 59, 59, tzinfo=timezone.utc)
time_step = '1d' 

# Initialize request tracking
request_times = deque(maxlen=API_DAILY_LIMIT)  

## CAMS API Configuration

The CAMS API requires user authentication via email. Here's how the email configuration works:

1. When you first run the notebook, you'll be prompted to enter your CAMS-registered email
2. The email is stored in a `.env` file in your project directory
3. For subsequent runs, the email is automatically loaded from the `.env` file
4. You can change the email anytime by modifying the `.env` file

### Environment Variables
- `CAMS_EMAIL`: Your registered CAMS email address
- Location: `.env` file in the project root
- Format: `CAMS_EMAIL=your.email@example.com`

> **Note**: Make sure you've registered your email at [CAMS Radiation Service](https://atmosphere.copernicus.eu/) before running the notebook.

In [3]:
# Initialize the CAMS client
# This will either:
# 1. Use the email from .env if it exists
# 2. Prompt you to enter your email if it's not configured
client = CAMSClient()

# You can check the current email configuration
import os
from dotenv import load_dotenv

load_dotenv()
current_email = os.getenv('CAMS_EMAIL')
if current_email:
    print(f"Currently configured email: {current_email}")
else:
    print("No email configured yet. You will be prompted for it when downloading data.")

Currently configured email: mukiibirogerz@gmail.com


## Create Grid Points for Uganda

The following cell will:
1. Load the Uganda shapefile
2. Create a grid of points covering Uganda
3. Filter points to only include those within Uganda's boundaries

In [4]:
# Load Uganda shapefile and prepare boundary
uganda_map = gpd.read_file(UGANDA_SHAPEFILE)
uganda_map = uganda_map.to_crs(epsg=4326)  # Convert to WGS84 coordinate system
uganda_boundary = uganda_map.union_all()
uganda_boundary = uganda_boundary.buffer(0)  # Fix any topology errors

# Get the bounding box of Uganda
minx, miny, maxx, maxy = uganda_boundary.bounds

# Create a grid of points
lon_grid = np.arange(minx, maxx, GRID_SPACING_DEGREES)
lat_grid = np.arange(miny, maxy, GRID_SPACING_DEGREES)
lons, lats = np.meshgrid(lon_grid, lat_grid)
grid_points = [(lon, lat) for lon, lat in zip(lons.flatten(), lats.flatten())]

# Filter points to only include those within Uganda
grid_point_objects = [Point(lon, lat) for lon, lat in grid_points]
grid_gdf = gpd.GeoDataFrame(geometry=grid_point_objects, crs=uganda_map.crs)
grid_gdf = grid_gdf[grid_gdf.within(uganda_boundary)]

# Extract final coordinates
uganda_coordinates = [(point.geometry.x, point.geometry.y) for _, point in grid_gdf.iterrows()]
print(f"Number of coordinates within Uganda boundary: {len(uganda_coordinates)}")

Number of coordinates within Uganda boundary: 1962


## Download CAMS Radiation Data

The following cell will download radiation data for each grid point:
- Checks if data for a coordinate has already been downloaded
- Downloads data for points that haven't been processed
- Saves data to a CSV file
- Includes a 30-second cooldown between requests

> **Note**: The process may take several hours due to API rate limits (100 requests per day)
> 
> **Tip**: You can safely interrupt the download at any time by pressing `Ctrl+C`. 
> Your progress will be saved, and you can resume from where you left off later.

In [5]:
# Initialize CAMS client (will prompt for email if not configured)
client = CAMSClient()

# Set up output file
csv_filename = "downloaded_data/CAMS_Radiation_ug_data.csv"
existing_coordinates, first_write = check_existing_coordinates(csv_filename)

print(f"Download period: {start_date.date()} to {end_date.date()} (UTC)")
print(f"Time step: {time_step}")
print(f"API Limits: {API_DAILY_LIMIT} requests per day")
print(f"Press Ctrl+C at any time to safely stop the download process")
print("-" * 40)

# Loop through each grid point and download data
skipped_count = 0
processed_count = 0

try:
    for i, point in enumerate(uganda_coordinates):
        lon, lat = point
        
        # Check if coordinate has already been processed
        if should_skip_coordinate(lat, lon, existing_coordinates):
            skipped_count += 1
            print(f"\nSkipping grid point {i+1}/{len(uganda_coordinates)}: (Lat={lat:.2f}, Lon={lon:.2f}) - Already downloaded")
            continue
        
        # Check API rate limit
        now = datetime.now(timezone.utc)
        if len(request_times) >= API_DAILY_LIMIT:
            oldest_request = request_times[0]
            time_since_oldest = now - oldest_request
            if time_since_oldest < timedelta(days=1):
                remaining_time = timedelta(days=1) - time_since_oldest
                print(f"\n⚠️ API daily limit reached. Please wait {remaining_time.total_seconds():.0f} seconds or resume tomorrow.")
                break
        
        processed_count += 1
        print(f"\nProcessing grid point {i+1}/{len(uganda_coordinates)}: (Lat={lat:.2f}, Lon={lon:.2f})")
        
        # Fetch data from CAMS
        result = client.fetch_data(
            latitude=lat,
            longitude=lon,
            start=start_date,
            end=end_date,
            time_step=time_step,
        )
        
        # Track this request
        request_times.append(now)
        
        if result["error"]:
            print(f"  ✗ FAILED. An error occurred: {result['error']}")
            continue
            
        # Process and save the data
        df = pd.DataFrame(result["data"])
        df['latitude'] = lat
        df['longitude'] = lon
        
        # Reorder columns to put coordinates first
        data_cols = [col for col in df.columns if col not in ['latitude', 'longitude']]
        df = df[['latitude', 'longitude'] + data_cols]
        
        # Save to CSV and update tracking
        df.to_csv(csv_filename, mode='a', header=first_write, index=False)
        add_coordinate_to_existing(lat, lon, existing_coordinates)
        first_write = False
        
        print(f"  ✓ Success! Fetched and wrote {len(df)} records to {csv_filename}.")
        
        # Add a small cooldown between requests
        if i < len(uganda_coordinates) - 1:  # Don't wait after the last point
            print(f"Cooling down for {API_COOLDOWN} seconds...")
            time.sleep(API_COOLDOWN)

except KeyboardInterrupt:
    print("\n\n📋 Download process interrupted by user")
    print("Don't worry - your progress has been saved!")
    print(f"You can restart the notebook later to continue from where you left off")

finally:
    # Print summary
    print("\n" + "="*40)
    print_download_summary(skipped_count, processed_count, csv_filename)
    show_data_preview(csv_filename, first_write)

    # Print remaining API capacity
    remaining_requests = API_DAILY_LIMIT - len(request_times)
    print(f"\nRemaining API capacity: {remaining_requests} requests for this 24-hour period")

Found existing CSV file with 7 unique coordinate pairs already downloaded.
Existing coordinates:
  (-1.3821, 29.6715)
  (-1.3821, 29.9715)
  (-1.2821, 29.6715)
  (-1.2821, 29.7715)
  (-1.2821, 29.8715)
  (-1.2821, 29.9715)
  (-1.2821, 30.0715)
Download period: 2024-01-01 to 2024-12-31 (UTC)
Time step: 1d
API Limits: 100 requests per day
Press Ctrl+C at any time to safely stop the download process
----------------------------------------

Skipping grid point 1/1962: (Lat=-1.38, Lon=29.67) - Already downloaded

Skipping grid point 2/1962: (Lat=-1.38, Lon=29.97) - Already downloaded

Skipping grid point 3/1962: (Lat=-1.28, Lon=29.67) - Already downloaded

Skipping grid point 4/1962: (Lat=-1.28, Lon=29.77) - Already downloaded

Skipping grid point 5/1962: (Lat=-1.28, Lon=29.87) - Already downloaded

Skipping grid point 6/1962: (Lat=-1.28, Lon=29.97) - Already downloaded

Skipping grid point 7/1962: (Lat=-1.28, Lon=30.07) - Already downloaded

Processing grid point 8/1962: (Lat=-1.18, Lon=2